Vào lệnh LONG khi thị trường xác nhận xu hướng tăng (MA9 > MA50), có lực mua đột phá (RSI > 40 và tăng ≥ 3 điểm) và được hậu thuẫn bởi dòng tiền lớn (Volume > trung bình 20 phiên).

Thoát lệnh bằng cơ chế Trailing Stop bám sát đỉnh, tự động cắt lệnh ngay khi giá sụt giảm vượt quá 2 lần biên độ dao động thực tế (2*ATR) tính từ mức đỉnh cao nhất đạt được.

In [1]:
!pip install pandas pandas-ta requests plotly -q

import requests
import time
import pandas as pd
import pandas_ta as ta
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from datetime import datetime

# ==========================================
# GLOBAL CONFIGURATION
# ==========================================
CONFIG = {
    'SYMBOLS': ['BTCUSDT', 'ETHUSDT'], # Đổi sang BTC, ETH cho chiến lược Trend Following
    'INTERVAL': '1h',
    'START_DATE': '01/01/2023',
    'END_DATE': '01/01/2024',
    'INITIAL_CAPITAL': 10000.0,
    'POSITION_SIZE_PCT': 0.10,
    'FEE_RATE': 0.0004,
    'SLIPPAGE': 0.0002
}

# ==========================================
# MODULE 1: DATA HANDLER
# ==========================================
class BinanceDataFetcher:
    def __init__(self, symbol, interval, start_date, end_date):
        self.symbol = symbol.upper()
        self.interval = interval
        self.start_date = start_date
        self.end_date = end_date

    def _str_to_millis(self, date_str: str) -> int:
        dt = datetime.strptime(date_str, "%d/%m/%Y")
        return int(dt.timestamp() * 1000)

    def fetch(self) -> pd.DataFrame:
        base_url = "https://data-api.binance.vision"
        req_symbol = self.symbol

        print(f"Checking connection for {req_symbol}...")
        try:
            test = requests.get(f"{base_url}/api/v3/time", timeout=5)
            if test.status_code == 451:
                print("US IP detected. Switching to Binance.US...")
                base_url = "https://api.binance.us"
                if req_symbol == "ETHUSDT": req_symbol = "ETHUSD"
                if req_symbol == "BNBUSDT": req_symbol = "BNBUSD"
                if req_symbol == "BTCUSDT": req_symbol = "BTCUSD"
        except Exception:
            base_url = "https://api.binance.us"
            if req_symbol == "ETHUSDT": req_symbol = "ETHUSD"
            if req_symbol == "BNBUSDT": req_symbol = "BNBUSD"
            if req_symbol == "BTCUSDT": req_symbol = "BTCUSD"

        url = base_url + "/api/v3/klines"
        start_time = self._str_to_millis(self.start_date)

        if self.end_date == datetime.now().strftime("%d/%m/%Y"):
            end_time = int(time.time() * 1000)
        else:
            end_dt = datetime.strptime(self.end_date, "%d/%m/%Y").replace(hour=23, minute=59, second=59)
            end_time = int(end_dt.timestamp() * 1000)

        all_klines =[]
        print(f"Downloading {req_symbol} ({self.interval})...")

        while start_time < end_time:
            params = {"symbol": req_symbol, "interval": self.interval, "startTime": start_time, "endTime": end_time, "limit": 1000}
            try:
                r = requests.get(url, params=params, timeout=10)
                if r.status_code != 200: break
                data = r.json()
                if not data: break
                all_klines.extend(data)
                start_time = data[-1][0] + 1
                time.sleep(0.05)
            except Exception: break

        if not all_klines: return pd.DataFrame()

        df = pd.DataFrame(all_klines, columns=["timestamp", "Open", "High", "Low", "Close", "Volume", "CT", "QAV", "NT", "TBBV", "TBQV", "I"])
        df = df[["timestamp", "Open", "High", "Low", "Close", "Volume"]]
        df["timestamp"] = pd.to_datetime(df["timestamp"], unit="ms")
        df.set_index("timestamp", inplace=True)
        df = df.astype(float)
        print(f"Total: {len(df)} candles fetched.")
        return df

# ==========================================
# MODULE 2: STRATEGY LOGIC (MA & RSI)
# ==========================================
class OptimisedRsiMaStrategy:
    def __init__(self):
        self.ma_fast = 9
        self.ma_slow = 50
        self.rsi_period = 14

        self.rsi_min = 40
        self.rsi_momentum = 3.0
        self.trail_pct = 0.02 # 2% Trailing Stop/TP

    def calculate_indicators(self, df: pd.DataFrame) -> pd.DataFrame:
        df['RSI'] = ta.rsi(df['Close'], length=self.rsi_period)
        df['MA9'] = ta.sma(df['Close'], length=self.ma_fast)
        df['MA50'] = ta.sma(df['Close'], length=self.ma_slow)

        df.dropna(inplace=True)
        return df

# ==========================================
# MODULE 3: BACKTEST ENGINE (LONG ONLY + TRAILING)
# ==========================================
class BacktestEngine:
    def __init__(self, df, strategy, config):
        self.df = strategy.calculate_indicators(df.copy())
        self.strategy = strategy
        self.config = config

        self.capital = config['INITIAL_CAPITAL']
        self.invested_amount = 0.0
        self.position = 0 # 0: Flat, 1: Long
        self.entry_price = 0.0
        self.trades = []
        self.equity_curve = np.zeros(len(self.df))

        # Trailing Stop Variables
        self.extreme_price = 0.0
        self.trail_stop_price = 0.0

    def run(self):
        O = self.df['Open'].values
        H = self.df['High'].values
        L = self.df['Low'].values
        C = self.df['Close'].values
        times = self.df.index

        rsi = self.df['RSI'].values
        ma9 = self.df['MA9'].values
        ma50 = self.df['MA50'].values

        fee = self.config['FEE_RATE']
        slip = self.config['SLIPPAGE']
        pos_size_pct = self.config['POSITION_SIZE_PCT']
        trail_pct = self.strategy.trail_pct

        s = self.strategy

        # Khởi tạo Equity ban đầu
        self.equity_curve[0] = self.capital
        self.equity_curve[1] = self.capital

        # Chạy từ index 2 vì cần rsi[i-2] để tính momentum
        for i in range(2, len(self.df)):

            # 1. EXIT LOGIC (2% TRAILING STOP)
            if self.position == 1:
                exit_signal = False

                # Cập nhật giá cao nhất đạt được (Peak)
                if H[i] > self.extreme_price:
                    self.extreme_price = H[i]

                # Tính mức Stoploss mới từ đỉnh
                current_trail = self.extreme_price * (1 - trail_pct)

                # SL chỉ được phép dời LÊN
                if current_trail > self.trail_stop_price or self.trail_stop_price == 0:
                    self.trail_stop_price = current_trail

                # Kiểm tra nến hiện tại có đâm xuống chạm Trailing Stop không
                if L[i] <= self.trail_stop_price:
                    exit_signal = True
                    # Nếu mở cửa nhảy Gap qua luôn SL, khớp ở giá mở cửa
                    raw_exit_price = min(O[i], self.trail_stop_price)

                if exit_signal:
                    # Bán ra nên trượt giá làm giá bán THẤP hơn
                    real_exit = raw_exit_price * (1 - slip)

                    pnl_pct = (real_exit - self.entry_price) / self.entry_price
                    net_pnl_pct = pnl_pct - (fee * 2)
                    pnl_amt = self.invested_amount * net_pnl_pct
                    self.capital += pnl_amt

                    self.trades.append({
                        'Date': times[i], 'Type': 'Long',
                        'Reason': f"Trailing Stop 2%",
                        'Entry': self.entry_price, 'Exit': real_exit,
                        'PnL_Amt': pnl_amt, 'Equity': self.capital
                    })

                    # Reset Variables
                    self.position = 0
                    self.invested_amount = 0.0
                    self.extreme_price = 0.0
                    self.trail_stop_price = 0.0

            # 2. ENTRY LOGIC (LONG ONLY)
            if self.position == 0:
                rsi_momentum = rsi[i-1] - rsi[i-2]

                cond_ma = ma9[i-1] > ma50[i-1]
                cond_rsi_level = rsi[i-1] > s.rsi_min
                cond_rsi_mom = rsi_momentum >= s.rsi_momentum

                if cond_ma and cond_rsi_level and cond_rsi_mom:
                    self.position = 1
                    # Mua vào nên trượt giá làm giá mua CAO hơn
                    self.entry_price = O[i] * (1 + slip)
                    self.invested_amount = self.capital * pos_size_pct

                    # Khởi tạo Trailing Stop ngay khi khớp lệnh
                    self.extreme_price = self.entry_price
                    self.trail_stop_price = self.extreme_price * (1 - trail_pct)

            # 3. UNREALIZED PNL CẬP NHẬT TRẠNG THÁI VỐN
            current_equity = self.capital
            if self.position == 1:
                # Với lệnh Long: (Giá hiện tại - Giá mua) / Giá mua
                unrealized_pnl_pct = (C[i] - self.entry_price) / self.entry_price
                unrealized_pnl = self.invested_amount * unrealized_pnl_pct
                current_equity += unrealized_pnl

            self.equity_curve[i] = current_equity

        return pd.Series(self.equity_curve, index=self.df.index), pd.DataFrame(self.trades)

# ==========================================
# MODULE 4: PERFORMANCE EVALUATOR
# ==========================================
class Evaluator:
    @staticmethod
    def generate_report(df_raw, equity, trades_df, initial_capital, symbol, interval, pos_size_pct):
        if trades_df.empty:
            print(f"No trades executed for {symbol}!")
            return

        final_equity = equity.iloc[-1]
        net_pnl = final_equity - initial_capital
        total_return = net_pnl / initial_capital

        rolling_max = equity.cummax()
        drawdown = (equity - rolling_max) / rolling_max
        max_dd = drawdown.min()

        wins = trades_df[trades_df['PnL_Amt'] > 0]
        losses = trades_df[trades_df['PnL_Amt'] <= 0]
        winrate = len(wins) / len(trades_df) if len(trades_df) > 0 else 0
        profit_factor = wins['PnL_Amt'].sum() / abs(losses['PnL_Amt'].sum()) if len(losses) > 0 and losses['PnL_Amt'].sum() != 0 else 0

        days = (equity.index[-1] - equity.index[0]).days
        cagr = ((final_equity / initial_capital) ** (365.0 / days)) - 1 if days > 0 else 0

        def get_annual_factor(tf):
            if tf == '15m': return 365 * 24 * 4
            elif tf == '1h': return 365 * 24
            elif tf == '4h': return 365 * 6
            elif tf == '1d': return 365
            return 365 * 24

        annual_factor = get_annual_factor(interval)
        returns = equity.pct_change().dropna()
        sharpe_ratio = (returns.mean() / returns.std()) * np.sqrt(annual_factor) if returns.std() != 0 else 0

        print("")
        print(f"========== PERFORMANCE REPORT: {symbol} ({interval}) ==========")
        print(f"{'Strategy:':>20} Optimised RSI & MA")
        print(f"{'Position Sizing:':>20} {pos_size_pct*100}% of Capital")
        print(f"{'Initial Capital:':>20} ${initial_capital:,.2f}")
        print(f"{'Final Equity:':>20} ${final_equity:,.2f}")
        print(f"{'Net PnL:':>20} ${net_pnl:,.2f}")
        print(f"{'Total Return:':>20} {total_return:.2%}")
        print(f"{'CAGR (Annual):':>20} {cagr:.2%}")
        print(f"{'Max Drawdown:':>20} {max_dd:.2%}")
        print(f"{'Sharpe Ratio:':>20} {sharpe_ratio:.2f}")
        print(f"{'Total Trades:':>20} {len(trades_df)}")
        print(f"{'Winrate:':>20} {winrate:.2%}")
        print(f"{'Profit Factor:':>20} {profit_factor:.2f}")
        print("====================================================\n")

        fig = make_subplots(
            rows=3, cols=1,
            shared_xaxes=True,
            vertical_spacing=0.05,
            row_heights=[0.5, 0.3, 0.2],
            subplot_titles=(f"Price Chart ({symbol})", "Equity Curve", "Drawdown")
        )

        fig.add_trace(go.Candlestick(
            x=df_raw.index, open=df_raw['Open'], high=df_raw['High'], low=df_raw['Low'], close=df_raw['Close'],
            name="Price", increasing_line_color='#26a69a', decreasing_line_color='#ef5350'
        ), row=1, col=1)

        fig.add_trace(go.Scatter(
            x=equity.index, y=equity.values,
            mode="lines", name="Equity",
            line=dict(color="#2962ff", width=1.5)
        ), row=2, col=1)

        fig.add_trace(go.Scatter(
            x=drawdown.index, y=drawdown.values,
            mode="lines", name="Drawdown",
            line=dict(color="#d32f2f", width=1),
            fill='tozeroy', fillcolor='rgba(211, 47, 47, 0.2)'
        ), row=3, col=1)

        fig.update_layout(
            title=f"Full Portfolio Performance: {symbol} ({interval})",
            template="plotly_white", height=900,
            margin=dict(l=40, r=40, t=60, b=40), showlegend=False,
            xaxis_rangeslider_visible=False
        )

        fig.update_xaxes(showgrid=True, gridcolor='#e6e6e6', zeroline=False)
        fig.update_yaxes(title_text="Price ($)", showgrid=True, gridcolor='#e6e6e6', row=1, col=1)
        fig.update_yaxes(title_text="Balance ($)", tickprefix="$", showgrid=True, gridcolor='#e6e6e6', zeroline=False, row=2, col=1)
        fig.update_yaxes(title_text="Drawdown %", tickformat=".1%", showgrid=True, gridcolor='#e6e6e6', zeroline=True, zerolinecolor='#cccccc', row=3, col=1)

        fig.show()

# ==========================================
# MAIN EXECUTION
# ==========================================
if __name__ == "__main__":
    for symbol in CONFIG['SYMBOLS']:
        print(f"\n---> Starting Backtest for: {symbol} <---")
        fetcher = BinanceDataFetcher(symbol, CONFIG['INTERVAL'], CONFIG['START_DATE'], CONFIG['END_DATE'])
        df_raw = fetcher.fetch()

        if not df_raw.empty:
            my_strategy = OptimisedRsiMaStrategy()
            engine = BacktestEngine(df_raw, my_strategy, CONFIG)
            equity, trades = engine.run()

            Evaluator.generate_report(
                df_raw, equity, trades,
                CONFIG['INITIAL_CAPITAL'], symbol,
                CONFIG['INTERVAL'], CONFIG['POSITION_SIZE_PCT']
            )

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 3.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 240.3/240.3 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 55.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 47.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.5/16.5 MB 51.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.4/42.4 MB 13.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.2 which is incompatible.
tensorflow 2.19.0 requires numpy<2.2.0,>=1.26.0, but you have numpy 2.2.6 which is incompatible.
db-dtypes 1.5.1 requires pandas<3.0.0,>=1.5.3, but you have pandas 3.0.2 which is 


---> Starting Backtest for: ETHUSDT <---
Checking connection for ETHUSDT...
Total: 8783 candles fetched.

========== PERFORMANCE REPORT: ETHUSDT (1h) ==========
           Strategy: Optimised RSI & MA
    Position Sizing: 10.0% of Capital
    Initial Capital: $10,000.00
       Final Equity: $8,923.28
            Net PnL: $-1,076.72
       Total Return: -10.77%
      CAGR (Annual): -10.82%
       Max Drawdown: -11.95%
       Sharpe Ratio: -3.79
       Total Trades: 266
            Winrate: 31.20%
      Profit Factor: 0.52

